Ejecuta un LLM (LLama-3.1-8B-Instruct) para determinar cual de los tipos de entidad extraidos por 

In [1]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
assert torch.backends.mps.is_available()
device = torch.device("mps")

/Users/diegolarraguibel/Desktop/Semestre 2025-2/ia generativa/Proyecto-IA-Gen/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
login("key")

In [14]:
tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct",
    torch_dtype=torch.float16, # Bajé a float16 para que quepa en memoria
    low_cpu_mem_usage=True
).to(device)

if model.generation_config.pad_token_id is None and tok.pad_token_id is None:
    model.generation_config.pad_token_id = tok.eos_token_id

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:11<00:00,  2.99s/it]


In [16]:
context = """Tu tarea consiste en seleccionar, para cada "entidad", la mejor etiqueta dentro de las opciones en "superclase" que la describa de forma más simple, clara y general.  

Debes elegir **una sola palabra o frase corta** que capture de manera eficiente la naturaleza de la entidad, evitando redundancias o términos demasiado específicos.  

---

### MODO DE RAZONAMIENTO (ejemplo a seguir)
1. Analiza qué tipo de cosa es la entidad (¿persona, lugar, organización, concepto?).
2. Revisa las superclases y determina cuál describe de forma más directa, simple y general.
3. Si varias opciones son posibles, prefiere:
   - la más general y común;
   - la que se entiende por sí sola sin contexto;
   - la que sería más útil como etiqueta en un grafo de conocimiento.
4. Evita términos demasiado específicos, técnicos o redundantes.

---

### EJEMPLOS CON RAZONAMIENTO

**Ejemplo 1**
entidad: Reino Unido  
superclase: país, país insular, estado soberano, poder colonial  
razonamiento:  
Reino Unido es un estado compuesto que cumple con las características de un país soberano. “País insular” y “poder colonial” son descripciones históricas o geográficas, pero “país” es la etiqueta más simple y general.  
output: país  

**Ejemplo 2**
entidad: Rafaela  
superclase: municipio, asentamiento, municipio de Argentina, ciudad de Argentina  
razonamiento:  
Rafaela es una localidad urbana dentro de Argentina. “Asentamiento” y “municipio” son más generales, pero “ciudad de Argentina” es la forma más específica y natural que la describe sin redundancia.  
output: ciudad de Argentina  

**Ejemplo 3**
entidad: empresario  
superclase: profesión, persona jurídica, ocupación, concepto económico, business and administration professionals  
razonamiento:  
“Empresario” se refiere a una persona que ejerce una actividad económica. No es una persona jurídica, sino una ocupación o rol laboral. “Ocupación” es la etiqueta más general y adecuada.  
output: ocupación  

**Ejemplo 4**
entidad: ingeniero  
superclase: profesión, cargo, trabajador  
razonamiento:  
“Profesión” describe directamente lo que es ser ingeniero, mientras que “cargo” o “trabajador” son categorías más amplias.  
output: profesión  

**Ejemplo 5**
entidad: pintor  
superclase: profesión, Q778000, trabajador de la construcción, menestral  
razonamiento:  
“Pintor” puede ser artístico o técnico, pero en ambos casos es una profesión. “Trabajador de la construcción” es un subconjunto y “menestral” es arcaico.  
output: profesión  

---

### NUEVO CASO

Ahora aplica el mismo razonamiento anterior, pero **sin mostrar el razonamiento**, solo entrega el resultado final.  

entidad: {{entidad}}  
superclase: {{lista_de_superclases}}  

output:
""".strip()

Tu tarea consiste en seleccionar, para cada "entidad", la mejor etiqueta dentro de las opciones en "superclase" que la describa de forma más simple, clara y general.  

Debes elegir **una sola palabra o frase corta** que capture de manera eficiente la naturaleza de la entidad, evitando redundancias o términos demasiado específicos.  

---

### MODO DE RAZONAMIENTO (ejemplo a seguir)
1. Analiza qué tipo de cosa es la entidad (¿persona, lugar, organización, concepto?).
2. Revisa las superclases y determina cuál describe de forma más directa, simple y general.
3. Si varias opciones son posibles, prefiere:
   - la más general y común;
   - la que se entiende por sí sola sin contexto;
   - la que sería más útil como etiqueta en un grafo de conocimiento.
4. Evita términos demasiado específicos, técnicos o redundantes.

---

### EJEMPLOS CON RAZONAMIENTO

**Ejemplo 1**
entidad: Reino Unido  
superclase: país, país insular, estado soberano, poder colonial  
razonamiento:  
Reino Unido es un es

In [15]:
messages = [{"role": "user", "content": "Yo bro, did I load the LLaMA 3 model correctly?"}]
inputs = tok.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt",
    return_dict=True
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

gen = model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

prompt_len = inputs["input_ids"].shape[-1]
print(tok.decode(gen[0, prompt_len:], skip_special_tokens=True))

It seems you're trying to load the LLaMA 3 model. However, I'm a large language model, I don't have direct access to your code or environment. 

To troubleshoot
